# Episode 5 — Dual-curve pricing

Companion notebook for the video. We forecast 3-month BBSW on one curve and discount on the AONIA curve, compare that with the single-curve set-up of Episode 4, and see where the difference shows up: not in par swaps, but in off-market swaps and in how risk splits between the two curves. Every number on screen or in the narration is produced here and read from `build/outputs.json`.

> **Illustrative data.** Both quote files are made up for teaching (the AONIA quotes from Episode 3, the BBSW quotes from Episode 4). They are **not market prices**. Educational material only, not investment advice.

1. Setup · 2. Why discount on AONIA · 3. The AONIA curve · 4. Two BBSW curves · 5. Forwards and discount factors · 6. Par swaps · 7. An off-market swap · 8. One cash flow by hand · 9. Which curve carries the risk · 10. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data files this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
#@title Data: writes `quotes_ois_illustrative.csv` (run me first) { display-mode: "form" }
# Illustrative quotes, made up for teaching. Not market data.
from pathlib import Path
Path('quotes_ois_illustrative.csv').parent.mkdir(parents=True, exist_ok=True)
Path('quotes_ois_illustrative.csv').write_text("""tenor,instrument,rate_pct,note
O/N,deposit,3.85,Stand-in for today's AONIA (not published until tomorrow): modelling choice
1M,ois,3.86,
3M,ois,3.89,
6M,ois,3.93,
9M,ois,3.97,
1Y,ois,4.00,
18M,ois,4.05,
2Y,ois,4.08,
3Y,ois,4.13,
5Y,ois,4.24,
7Y,ois,4.35,
10Y,ois,4.50,
""")
print('wrote quotes_ois_illustrative.csv')

In [ ]:
#@title Data: writes `quotes_bbsw_illustrative.csv` (run me first) { display-mode: "form" }
# Illustrative quotes, made up for teaching. Not market data.
from pathlib import Path
Path('quotes_bbsw_illustrative.csv').parent.mkdir(parents=True, exist_ok=True)
Path('quotes_bbsw_illustrative.csv').write_text("""tenor,instrument,rate_pct
3M,BBSW,4.02
3x6,FRA,4.08
6x9,FRA,4.13
9x12,FRA,4.17
12x15,FRA,4.20
15x18,FRA,4.23
18x21,FRA,4.25
2Y,Swap,4.17
3Y,Swap,4.22
""")
print('wrote quotes_bbsw_illustrative.csv')

In [ ]:
# Parameters (papermill overrides these)
valuation_date = "2026-09-22"
ois_quotes_file = "quotes_ois_illustrative.csv"
bbsw_quotes_file = "quotes_bbsw_illustrative.csv"
output_json = "build/outputs.json"
notional = 100_000_000
offmarket_fixed_pct = 4.50

## 1. Setup

In [ ]:
import json
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import QuantLib as ql

today = ql.DateParser.parseISO(valuation_date)
ql.Settings.instance().evaluationDate = today
dc = ql.Actual365Fixed()
iso = lambda d: d.ISO()

cal = ql.Australia(ql.Australia.Settlement)
# QuantLib 1.43 misses NSW's additional Anzac Day holidays when 25 April falls on a weekend.
for d in [ql.Date(27, 4, 2026), ql.Date(26, 4, 2027)]:
    cal.addHoliday(d)
spot = cal.advance(today, 1, ql.Days)

def aonia_index(curve=ql.YieldTermStructureHandle()):
    return ql.OvernightIndex("AONIA", 0, ql.AUDCurrency(), cal, dc, curve)

def bbsw3m(curve=ql.YieldTermStructureHandle()):
    return ql.IborIndex("BBSW3M", ql.Period(3, ql.Months), 0, ql.AUDCurrency(), cal,
                        ql.ModifiedFollowing, False, dc, curve)

ois_quotes = pd.read_csv(ois_quotes_file).fillna("")
bbsw_quotes = pd.read_csv(bbsw_quotes_file)
out = {"meta": {
    "episode": 5, "valuation_date": valuation_date, "quantlib_version": ql.__version__,
    "quotes_label": "ILLUSTRATIVE - not market data",
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}
print("QuantLib", ql.__version__, "| valuation date", today)

## 2. Why discount on AONIA

Cleared and collateralised swaps post cash collateral that earns an overnight rate, so their cash flows are discounted on the overnight curve. Clearing houses set the interest they pay on collateral and their discount curve together (see LCH's 2020 filing moving both PAI and discounting to SOFR and €STR). For AUD, AFMA §3.6: prices for uncleared derivatives are quoted as if discounted like cleared ones, which *"in the case of AUD swaps currently"* means AUD OIS curves.

BBSW is still what the floating leg pays. So we need two curves: **AONIA to discount**, **BBSW to forecast**.

## 3. The AONIA curve (as in Episode 3)

In [ ]:
def build_ois_curve(bump_bp=0.0):
    idx = aonia_index()
    helpers = []
    for row in ois_quotes.itertuples():
        q = ql.QuoteHandle(ql.SimpleQuote(row.rate_pct / 100 + bump_bp / 1e4))
        if row.instrument == "deposit":
            helpers.append(ql.DepositRateHelper(q, ql.Period(1, ql.Days), 0, cal, ql.Following, False, dc))
        else:
            helpers.append(ql.OISRateHelper(1, ql.Period(row.tenor), q, idx, paymentLag=2, paymentFrequency=ql.Annual,
                                            paymentCalendar=cal, convention=ql.ModifiedFollowing, endOfMonth=False))
    c = ql.PiecewiseLogLinearDiscount(today, helpers, dc)
    c.enableExtrapolation()
    return c

ois_curve = build_ois_curve()
ois_handle = ql.YieldTermStructureHandle(ois_curve)
print("AONIA 3Y discount factor:", ois_curve.discount(cal.advance(spot, 3, ql.Years)))

## 4. Two BBSW curves

* **Single curve** (Episode 4): the BBSW curve forecasts and discounts.
* **Dual curve**: the swap helpers get the AONIA curve as `discountingCurve`, so the bootstrap solves only for BBSW forwards. The fixing and the FRAs don't depend on discounting.

In [ ]:
def build_bbsw_curve(discount=None, bump_bp=0.0):
    index = bbsw3m()
    discounting = discount if discount is not None else ql.YieldTermStructureHandle()  # empty = one curve
    helpers = []
    for row in bbsw_quotes.itertuples():
        q = ql.QuoteHandle(ql.SimpleQuote(row.rate_pct / 100 + bump_bp / 1e4))
        if row.instrument == "BBSW":
            h = ql.DepositRateHelper(q, index)
        elif row.instrument == "FRA":
            h = ql.FraRateHelper(q, int(row.tenor.split("x")[0]), index)
        else:
            h = ql.SwapRateHelper(q, ql.Period(row.tenor), cal, ql.Quarterly,
                                  ql.ModifiedFollowing, dc, index, ql.QuoteHandle(),
                                  ql.Period(0, ql.Days), discounting, 1)
        helpers.append(h)
    c = ql.PiecewiseLogLinearDiscount(today, helpers, dc)
    c.enableExtrapolation()
    return c

single = build_bbsw_curve()                        # forecasts and discounts
dual = build_bbsw_curve(discount=ois_handle)       # forecasts only; AONIA discounts
single_h, dual_h = ql.YieldTermStructureHandle(single), ql.YieldTermStructureHandle(dual)

## 5. Forwards and discount factors

In [ ]:
def fwd3m(curve, d0):
    d1 = cal.advance(d0, 3, ql.Months, ql.ModifiedFollowing, False)
    return curve.forwardRate(d0, d1, dc, ql.Simple).rate() * 100

grid = []
for m in range(0, 34):  # forward windows that end within the 3-year curve
    d0 = cal.advance(today, m, ql.Months)
    grid.append({"t_years": dc.yearFraction(today, d0), "bbsw_dual_pct": fwd3m(dual, d0),
                 "bbsw_single_pct": fwd3m(single, d0), "aonia_pct": fwd3m(ois_curve, d0)})
g = pd.DataFrame(grid)
g["spread_bp"] = (g.bbsw_dual_pct - g.aonia_pct) * 100
g["dual_minus_single_bp"] = (g.bbsw_dual_pct - g.bbsw_single_pct) * 100
out["forwards"] = {"grid": g.to_dict("records"),
                   "max_abs_dual_minus_single_bp": float(g.dual_minus_single_bp.abs().max()),
                   "spread_min_bp": float(g.spread_bp.min()), "spread_max_bp": float(g.spread_bp.max()),
                   "spread_avg_bp": float(g.spread_bp.mean()),
                   "ticks": [{"x": i / 2, "label": lab} for i, lab in enumerate(["0", "6M", "1Y", "18M", "2Y", "30M"])]}

dfs = []
for y in (1, 2, 3):
    d = cal.advance(spot, y, ql.Years)
    dfs.append({"tenor": f"{y}Y", "date": iso(d), "bbsw_single": single.discount(d), "aonia": ois_curve.discount(d)})
df_tab = pd.DataFrame(dfs)
df_tab["diff"] = df_tab.aonia - df_tab.bbsw_single
out["discount_factors"] = df_tab.to_dict("records")

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(g.t_years, g.bbsw_dual_pct, label="3M BBSW, dual curve")
ax.plot(g.t_years, g.bbsw_single_pct, "--", label="3M BBSW, single curve")
ax.plot(g.t_years, g.aonia_pct, label="3M AONIA")
ax.set(xlabel="years", ylabel="%"); ax.grid(alpha=.3); ax.legend()
print(out["forwards"]["max_abs_dual_minus_single_bp"], out["forwards"]["spread_avg_bp"])
df_tab

## 6. Par swaps

Price the 3-year swap at its quoted rate both ways. Both builds were calibrated to that quote, so both give a par rate equal to the quote and a value of zero.

In [ ]:
def make_swap(tenor, fixed_rate, forecast, discount, receive=True):
    end = cal.advance(spot, ql.Period(tenor), ql.ModifiedFollowing, False)
    sched = ql.Schedule(spot, end, ql.Period(ql.Quarterly), cal, ql.ModifiedFollowing, ql.ModifiedFollowing,
                        ql.DateGeneration.Forward, False)
    side = ql.VanillaSwap.Receiver if receive else ql.VanillaSwap.Payer
    sw = ql.VanillaSwap(side, notional, sched, fixed_rate, dc, sched, bbsw3m(forecast), 0.0, dc)
    sw.setPricingEngine(ql.DiscountingSwapEngine(discount))
    return sw

quote_3y = float(bbsw_quotes.set_index("tenor").rate_pct["3Y"]) / 100
par_single = make_swap("3Y", quote_3y, single_h, single_h)
par_dual = make_swap("3Y", quote_3y, dual_h, ois_handle)
out["par"] = {"quote_pct": quote_3y * 100,
              "par_single_pct": par_single.fairRate() * 100, "par_dual_pct": par_dual.fairRate() * 100,
              "npv_single": par_single.NPV(), "npv_dual": par_dual.NPV()}

# With two curves the floating leg no longer telescopes to P(t0) - P(tn) on the discount curve.
p0, pn = ois_curve.discount(spot), ois_curve.discount(par_dual.fixedLeg()[len(par_dual.fixedLeg()) - 1].date())
out["telescoping"] = {"pv_float_dual": abs(par_dual.floatingLegNPV()), "n_times_p0_minus_pn": notional * (p0 - pn)}
out["telescoping"]["gap"] = out["telescoping"]["pv_float_dual"] - out["telescoping"]["n_times_p0_minus_pn"]
pd.Series({**out["par"], **out["telescoping"]})

## 7. An off-market swap

Receive $K$ for three years. In both set-ups the value is $N (K - S) A$, but the annuity $A$ is discounted on different curves.

In [ ]:
K = offmarket_fixed_pct / 100
off_single = make_swap("3Y", K, single_h, single_h)
off_dual = make_swap("3Y", K, dual_h, ois_handle)
annuity = lambda sw: abs(sw.fixedLegBPS()) / (notional * 1e-4)
off = {"fixed_rate_pct": offmarket_fixed_pct,
       "annuity_single": annuity(off_single), "annuity_dual": annuity(off_dual),
       "npv_single": off_single.NPV(), "npv_dual": off_dual.NPV()}
off["npv_dual_hand"] = notional * (K - out["par"]["par_dual_pct"] / 100) * off["annuity_dual"]
off["diff"] = off["npv_dual"] - off["npv_single"]
out["offmarket"] = off
pd.Series(off)

## 8. One cash flow by hand

The first floating coupon of the dual-curve swap: the 3-month BBSW forward from the BBSW curve for the first period, times the year fraction and the notional, discounted with the AONIA discount factor at the payment date.

In [ ]:
c0 = ql.as_floating_rate_coupon(off_dual.floatingLeg()[0])
s0, e0, p0d = c0.accrualStartDate(), c0.accrualEndDate(), c0.date()
fwd_hand = (dual.discount(s0) / dual.discount(e0) - 1) / dc.yearFraction(s0, e0)
amount_hand = notional * fwd_hand * dc.yearFraction(s0, e0)
hand = {"notional": notional, "start": iso(s0), "end": iso(e0), "pay": iso(p0d), "days": c0.accrualDays(),
        "fwd_pct": fwd_hand * 100, "fwd_ql_pct": c0.indexFixing() * 100,
        "amount": amount_hand, "amount_ql": c0.amount(),
        "df_ois": ois_curve.discount(p0d), "pv": amount_hand * ois_curve.discount(p0d),
        "pv_ql": c0.amount() * ois_curve.discount(p0d)}
out["hand"] = hand
pd.Series(hand)

## 9. Which curve carries the risk?

Bump one set of quotes by +1bp at a time, rebuild **both** curves (the BBSW curve depends on the AONIA curve through discounting), and reprice the par swap and the off-market swap.

In [ ]:
def npv_with(ois_bp, bbsw_bp, fixed_rate):
    oc = build_ois_curve(ois_bp)
    oh = ql.YieldTermStructureHandle(oc)
    bc = build_bbsw_curve(discount=oh, bump_bp=bbsw_bp)
    return make_swap("3Y", fixed_rate, ql.YieldTermStructureHandle(bc), oh).NPV()

rows = []
for name, ob, bb in [("AONIA quotes +1bp", 1, 0), ("BBSW quotes +1bp", 0, 1), ("Both +1bp", 1, 1)]:
    rows.append({"bump": name,
                 "par_swap": npv_with(ob, bb, quote_3y) - npv_with(0, 0, quote_3y),
                 "offmarket_swap": npv_with(ob, bb, K) - npv_with(0, 0, K)})
risk = pd.DataFrame(rows)
out["risk"] = {"rows": risk.to_dict("records"),
               "par_ois": rows[0]["par_swap"], "par_bbsw": rows[1]["par_swap"], "par_both": rows[2]["par_swap"],
               "off_ois": rows[0]["offmarket_swap"], "off_bbsw": rows[1]["offmarket_swap"], "off_both": rows[2]["offmarket_swap"]}
risk

## 10. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")